In [1]:
from pathlib import Path
import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [3]:
train_transform = A.Compose([

    A.Resize(224,224),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.Rotate(limit=15, p=0.5),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

val_transform = A.Compose([

    A.Resize(224,224),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [5]:
class CattleDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.class_names = sorted([
            folder.name
            for folder in self.root_dir.iterdir()
            if folder.is_dir()
        ])

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.class_names)
        }

        for cls_name in self.class_names:

            cls_folder = self.root_dir / cls_name

            for img_path in cls_folder.glob("*"):

                self.image_paths.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls_name]
                )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [6]:
train_dataset = CattleDataset(
    root_dir="../datasets/identity_split/train",
    transform=train_transform
)

val_dataset = CattleDataset(
    root_dir="../datasets/identity_split/val",
    transform=val_transform
)

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [8]:
import math

class ArcFace(nn.Module):

    def __init__(self,
                 in_features,
                 out_features,
                 s=30.0,
                 m=0.50):

        super().__init__()

        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.FloatTensor(
                out_features,
                in_features
            )
        )

        nn.init.xavier_uniform_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings
        )

        weights = F.normalize(
            self.weight
        )

        cosine = F.linear(
            embeddings,
            weights
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_logits = torch.cos(
            theta + self.m
        )

        one_hot = F.one_hot(
            labels,
            num_classes=weights.shape[0]
        ).float()

        logits = (
            one_hot * target_logits
            +
            (1.0 - one_hot) * cosine
        )

        logits *= self.s

        return logits

In [9]:
class ArcFaceResNet50(nn.Module):

    def __init__(self,
                 num_classes):

        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = (
            backbone.fc.in_features
        )

        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.embedding = nn.Linear(
            in_features,
            512
        )

        self.arcface = ArcFace(
            512,
            num_classes
        )

    def forward(
        self,
        images,
        labels=None
    ):

        features = self.backbone(
            images
        )

        embeddings = self.embedding(
            features
        )

        embeddings = F.normalize(
            embeddings
        )

        if labels is None:
            return embeddings

        logits = self.arcface(
            embeddings,
            labels
        )

        return logits, embeddings

In [10]:
num_classes = len(
    train_dataset.class_names
)

model = ArcFaceResNet50(
    num_classes=num_classes
)

model = model.to(device)

print("Classes:", num_classes)

Classes: 84


In [11]:
model.load_state_dict(
    torch.load(
        "../models/cattle_arcface.pth",
        weights_only=True
    )
)

model.eval()

print("ArcFace model loaded.")

ArcFace model loaded.


In [12]:
embedding_database = []
label_database = []

with torch.no_grad():

    for images, labels in train_loader:

        images = images.to(device)

        embeddings = model(images)

        embedding_database.append(
            embeddings.cpu()
        )

        label_database.append(
            labels
        )

embedding_database = torch.cat(
    embedding_database
)

label_database = torch.cat(
    label_database
)

print(
    embedding_database.shape
)

print(
    label_database.shape
)

torch.Size([838, 512])
torch.Size([838])


In [13]:
import torch.nn.functional as F

def identify_cow(
    query_embedding,
    embedding_database,
    label_database
):

    similarities = F.cosine_similarity(
        query_embedding.unsqueeze(0),
        embedding_database
    )

    best_match_idx = torch.argmax(
        similarities
    )

    predicted_label = (
        label_database[
            best_match_idx
        ].item()
    )

    similarity = (
        similarities[
            best_match_idx
        ].item()
    )

    return (
        predicted_label,
        similarity
    )

In [14]:
query_image, query_label = (
    train_dataset[50]
)

query_tensor = (
    query_image
    .unsqueeze(0)
    .to(device)
)

with torch.no_grad():

    query_embedding = model(
        query_tensor
    )

query_embedding = (
    query_embedding.cpu()
)

predicted_label, similarity = (
    identify_cow(
        query_embedding.squeeze(0),
        embedding_database,
        label_database
    )
)

print(
    "Actual Label:",
    query_label
)

print(
    "Predicted Label:",
    predicted_label
)

print(
    "Similarity:",
    similarity
)

Actual Label: 5
Predicted Label: 5
Similarity: 0.9993360042572021


In [15]:
same_indices = []

for i in range(len(label_database)):
    for j in range(i + 1, len(label_database)):

        if label_database[i] == label_database[j]:

            same_indices.append((i, j))
            break

    if len(same_indices) > 0:
        break

print(same_indices)

[(0, 324)]


In [16]:
i, j = same_indices[0]

same_similarity = F.cosine_similarity(
    embedding_database[i].unsqueeze(0),
    embedding_database[j].unsqueeze(0)
)

print(
    "Same Cow Similarity:",
    same_similarity.item()
)

Same Cow Similarity: 0.9021157622337341


In [17]:
different_indices = []

for i in range(len(label_database)):
    for j in range(i + 1, len(label_database)):

        if label_database[i] != label_database[j]:

            different_indices.append((i, j))
            break

    if len(different_indices) > 0:
        break

print(different_indices)

[(0, 1)]


In [18]:
i, j = different_indices[0]

different_similarity = F.cosine_similarity(
    embedding_database[i].unsqueeze(0),
    embedding_database[j].unsqueeze(0)
)

print(
    "Different Cow Similarity:",
    different_similarity.item()
)

Different Cow Similarity: 0.04895825684070587
